In [1]:
# ============================================================
# DEEPSHIELD-AI
# VIDEO AI — ENVIRONMENT SETUP
# ============================================================

import os
import sys
import random

import numpy as np
import torch
import torchvision
import cv2

print("=" * 65)
print("DEEPSHIELD-AI — VIDEO AI")
print("ENVIRONMENT CHECK")
print("=" * 65)

print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("OpenCV     :", cv2.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

    print("GPU        :", torch.cuda.get_device_name(0))
    print(
        "CUDA       :",
        torch.version.cuda
    )

else:

    DEVICE = torch.device("cpu")

    print("GPU        : Not available")

print("Device     :", DEVICE)

DEEPSHIELD-AI — VIDEO AI
ENVIRONMENT CHECK
Python     : 3.12.10
PyTorch    : 2.12.1+cu130
Torchvision: 0.27.1+cu130
OpenCV     : 5.0.0

CUDA available: True
GPU        : NVIDIA GeForce RTX 2050
CUDA       : 13.0
Device     : cuda


In [2]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)
print("Reproducibility configuration complete.")

Random seed: 42
Reproducibility configuration complete.


In [3]:
# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = os.path.abspath(
    os.path.join(
        os.getcwd(),
        "..",
        "..",
        ".."
    )
)

DATASET_ROOT = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "video"
)

MODEL_ROOT = os.path.join(
    PROJECT_ROOT,
    "models",
    "video"
)

OUTPUT_ROOT = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "video"
)

os.makedirs(DATASET_ROOT, exist_ok=True)
os.makedirs(MODEL_ROOT, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("=" * 65)
print("PROJECT PATHS")
print("=" * 65)

print("Project root :", PROJECT_ROOT)
print("Dataset root :", DATASET_ROOT)
print("Model root   :", MODEL_ROOT)
print("Output root  :", OUTPUT_ROOT)

PROJECT PATHS
Project root : d:\
Dataset root : d:\datasets\video
Model root   : d:\models\video
Output root  : d:\outputs\video


In [4]:
# ============================================================
# DEEPSHIELD-AI — DATASET LIBRARY
# ============================================================

from datasets import load_dataset

print("Hugging Face Datasets imported successfully.")

Hugging Face Datasets imported successfully.


In [5]:
# ============================================================
# DEEPSHIELD-AI — LOAD SDFVD DATASET
# ============================================================

SDFVD_NAME = "Hemgg/SDFVD-video-dataset"

print("=" * 65)
print("LOADING SDFVD VIDEO DATASET")
print("=" * 65)

sdfvd_dataset = load_dataset(
    SDFVD_NAME,
    split="train"
)

print("\nDataset loaded successfully.")
print("Total videos:", len(sdfvd_dataset))
print("Columns     :", sdfvd_dataset.column_names)
print("Features    :", sdfvd_dataset.features)

LOADING SDFVD VIDEO DATASET


Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/106 [00:00<?, ?it/s]


Dataset loaded successfully.
Total videos: 106
Columns     : ['video', 'label']
Features    : {'video': Video(decode=True, stream_index=None, dimension_order='NCHW', num_ffmpeg_threads=1, device='cpu', seek_mode='exact'), 'label': ClassLabel(names=['Fake', 'Real'])}


In [7]:
# ============================================================
# DEEPSHIELD-AI — VERIFY SDFVD STRUCTURE
# WITHOUT VIDEO DECODING
# ============================================================

print("=" * 65)
print("SDFVD DATASET STRUCTURE")
print("=" * 65)

print("Columns:")
for column in sdfvd_dataset.column_names:
    print(" -", column)

print("\nFeatures:")
print(sdfvd_dataset.features)

print("\nVideo feature configuration:")

video_feature = sdfvd_dataset.features["video"]

print("Type:", type(video_feature))
print("Decode:", video_feature.decode)

print("\nDataset metadata check complete.")
print("Video decoding will be handled by OpenCV later.")

SDFVD DATASET STRUCTURE
Columns:
 - video
 - label

Features:
{'video': Video(decode=True, stream_index=None, dimension_order='NCHW', num_ffmpeg_threads=1, device='cpu', seek_mode='exact'), 'label': ClassLabel(names=['Fake', 'Real'])}

Video feature configuration:
Type: <class 'datasets.features.video.Video'>
Decode: True

Dataset metadata check complete.
Video decoding will be handled by OpenCV later.


In [8]:
# ============================================================
# DEEPSHIELD-AI — LABEL VERIFICATION
# WITHOUT VIDEO DECODING
# ============================================================

from collections import Counter

print("=" * 65)
print("SDFVD LABEL VERIFICATION")
print("=" * 65)

# Read only the label column
labels = sdfvd_dataset["label"]

label_counts = Counter(
    int(label) for label in labels
)

print("\nLabel counts:")

for label in sorted(label_counts):

    if label == 0:
        name = "FAKE"
    elif label == 1:
        name = "REAL"
    else:
        name = f"UNKNOWN-{label}"

    print(
        f"Label {label} ({name}): "
        f"{label_counts[label]}"
    )

print("\nTotal:", len(labels))

SDFVD LABEL VERIFICATION

Label counts:
Label 0 (FAKE): 53
Label 1 (REAL): 53

Total: 106


In [9]:
# ============================================================
# DEEPSHIELD-AI — VIDEO METADATA CHECK
# NO VIDEO DECODING
# ============================================================

print("=" * 65)
print("SDFVD VIDEO METADATA CHECK")
print("=" * 65)

video_column = sdfvd_dataset.data.column("video")

print("Video column type:")
print(type(video_column))

print("\nNumber of video records:")
print(len(video_column))

print("\nFirst raw video record:")
print(video_column[0])

SDFVD VIDEO METADATA CHECK
Video column type:
<class 'pyarrow.lib.ChunkedArray'>

Number of video records:
106

First raw video record:
[('bytes', None), ('path', 'C:\\Users\\saksh\\.cache\\huggingface\\hub\\datasets--Hemgg--SDFVD-video-dataset\\snapshots\\11239a51248ad96a767460b7613cbf3b99b2f547\\Fake\\vs1.mp4')]


In [10]:
# ============================================================
# DEEPSHIELD-AI — RAW VIDEO RECORD INSPECTION
# ============================================================

raw_video = video_column[0]

print("=" * 65)
print("RAW VIDEO RECORD")
print("=" * 65)

print("Type:", type(raw_video))

if hasattr(raw_video, "as_py"):
    raw_video = raw_video.as_py()

print("Python type:", type(raw_video))
print("Value:", raw_video)

if isinstance(raw_video, dict):
    print("\nKeys:")
    for key in raw_video.keys():
        print(" -", key)

RAW VIDEO RECORD
Type: <class 'pyarrow.lib.StructScalar'>
Python type: <class 'dict'>
Value: {'bytes': None, 'path': 'C:\\Users\\saksh\\.cache\\huggingface\\hub\\datasets--Hemgg--SDFVD-video-dataset\\snapshots\\11239a51248ad96a767460b7613cbf3b99b2f547\\Fake\\vs1.mp4'}

Keys:
 - bytes
 - path


In [11]:
# ============================================================
# DEEPSHIELD-AI — DATASET CACHE LOCATION
# ============================================================

print("=" * 65)
print("DATASET CACHE INFORMATION")
print("=" * 65)

print("Dataset cache directory:")
print(sdfvd_dataset.cache_files)

DATASET CACHE INFORMATION
Dataset cache directory:
[{'filename': 'C:\\Users\\saksh\\.cache\\huggingface\\datasets\\Hemgg___sdfvd-video-dataset\\default\\0.0.0\\11239a51248ad96a767460b7613cbf3b99b2f547\\sdfvd-video-dataset-train.arrow'}]


In [12]:
# ============================================================
# DEEPSHIELD-AI — LABEL / INDEX VERIFICATION
# ============================================================

print("=" * 65)
print("LABEL INDEX VERIFICATION")
print("=" * 65)

for i in range(10):

    label = int(sdfvd_dataset["label"][i])

    label_name = (
        "FAKE"
        if label == 0
        else "REAL"
    )

    print(
        f"Index {i:3d} | "
        f"Label {label} | "
        f"{label_name}"
    )

LABEL INDEX VERIFICATION
Index   0 | Label 0 | FAKE
Index   1 | Label 0 | FAKE
Index   2 | Label 0 | FAKE
Index   3 | Label 0 | FAKE
Index   4 | Label 0 | FAKE
Index   5 | Label 0 | FAKE
Index   6 | Label 0 | FAKE
Index   7 | Label 0 | FAKE
Index   8 | Label 0 | FAKE
Index   9 | Label 0 | FAKE


In [13]:
# ============================================================
# DEEPSHIELD-AI — VIDEO METADATA CHECK
# ============================================================

print("=" * 65)
print("SDFVD VIDEO METADATA CHECK")
print("=" * 65)

video_column = sdfvd_dataset.data.column("video")

print("Video column type:")
print(type(video_column))

print("\nNumber of video records:")
print(len(video_column))

print("\nFirst raw video record:")
print(video_column[0])

SDFVD VIDEO METADATA CHECK
Video column type:
<class 'pyarrow.lib.ChunkedArray'>

Number of video records:
106

First raw video record:
[('bytes', None), ('path', 'C:\\Users\\saksh\\.cache\\huggingface\\hub\\datasets--Hemgg--SDFVD-video-dataset\\snapshots\\11239a51248ad96a767460b7613cbf3b99b2f547\\Fake\\vs1.mp4')]


In [14]:
# ============================================================
# DEEPSHIELD-AI — RAW VIDEO RECORD INSPECTION
# ============================================================

raw_video = video_column[0]

print("=" * 65)
print("RAW VIDEO RECORD")
print("=" * 65)

print("Type:", type(raw_video))

if hasattr(raw_video, "as_py"):
    raw_video = raw_video.as_py()

print("Python type:", type(raw_video))
print("Value:", raw_video)

if isinstance(raw_video, dict):
    print("\nKeys:")
    for key in raw_video.keys():
        print(" -", key)


RAW VIDEO RECORD
Type: <class 'pyarrow.lib.StructScalar'>
Python type: <class 'dict'>
Value: {'bytes': None, 'path': 'C:\\Users\\saksh\\.cache\\huggingface\\hub\\datasets--Hemgg--SDFVD-video-dataset\\snapshots\\11239a51248ad96a767460b7613cbf3b99b2f547\\Fake\\vs1.mp4'}

Keys:
 - bytes
 - path


In [15]:
# ============================================================
# DEEPSHIELD-AI — VERIFY ACTUAL VIDEO FILES
# ============================================================

print("=" * 65)
print("SDFVD VIDEO FILE VERIFICATION")
print("=" * 65)

valid_videos = 0
missing_videos = 0

for i in range(len(sdfvd_dataset)):

    raw_video = sdfvd_dataset.data.column("video")[i].as_py()

    video_path = raw_video["path"]

    if video_path and os.path.isfile(video_path):
        valid_videos += 1
    else:
        missing_videos += 1

print("Valid video files   :", valid_videos)
print("Missing video files :", missing_videos)
print("Total               :", len(sdfvd_dataset))

SDFVD VIDEO FILE VERIFICATION
Valid video files   : 106
Missing video files : 0
Total               : 106


In [16]:
# ============================================================
# DEEPSHIELD-AI — OPENCV VIDEO CHECK
# ============================================================

print("=" * 65)
print("OPENCV VIDEO CHECK")
print("=" * 65)

for i in range(3):

    raw_video = sdfvd_dataset.data.column("video")[i].as_py()
    video_path = raw_video["path"]

    cap = cv2.VideoCapture(video_path)

    opened = cap.isOpened()

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = cap.get(
        cv2.CAP_PROP_FPS
    )

    width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    cap.release()

    print(f"\nVideo {i}")
    print("Path        :", video_path)
    print("Opened      :", opened)
    print("Frames      :", frame_count)
    print("FPS         :", fps)
    print("Resolution  :", width, "x", height)

OPENCV VIDEO CHECK

Video 0
Path        : C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547\Fake\vs1.mp4
Opened      : True
Frames      : 82
FPS         : 30.0
Resolution  : 1280 x 720

Video 1
Path        : C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547\Fake\vs10.mp4
Opened      : True
Frames      : 110
FPS         : 30.0
Resolution  : 378 x 720

Video 2
Path        : C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547\Fake\vs11.mp4
Opened      : True
Frames      : 82
FPS         : 30.0
Resolution  : 1280 x 674


In [17]:
# ============================================================
# DEEPSHIELD-AI — CREATE VIDEO METADATA
# ============================================================

video_metadata = []

for i in range(len(sdfvd_dataset)):

    raw_video = sdfvd_dataset.data.column("video")[i].as_py()

    video_path = raw_video["path"]
    label = int(sdfvd_dataset["label"][i])

    video_metadata.append({
        "index": i,
        "video_path": video_path,
        "label": label
    })

print("=" * 65)
print("VIDEO METADATA CREATED")
print("=" * 65)

print("Total videos:", len(video_metadata))

print("\nFirst record:")
print(video_metadata[0])

print("\nLast record:")
print(video_metadata[-1])

VIDEO METADATA CREATED
Total videos: 106

First record:
{'index': 0, 'video_path': 'C:\\Users\\saksh\\.cache\\huggingface\\hub\\datasets--Hemgg--SDFVD-video-dataset\\snapshots\\11239a51248ad96a767460b7613cbf3b99b2f547\\Fake\\vs1.mp4', 'label': 0}

Last record:
{'index': 105, 'video_path': 'C:\\Users\\saksh\\.cache\\huggingface\\hub\\datasets--Hemgg--SDFVD-video-dataset\\snapshots\\11239a51248ad96a767460b7613cbf3b99b2f547\\Real\\v9.mp4', 'label': 1}


In [18]:
# ============================================================
# DEEPSHIELD-AI — METADATA LABEL CHECK
# ============================================================

metadata_fake = sum(
    item["label"] == 0
    for item in video_metadata
)

metadata_real = sum(
    item["label"] == 1
    for item in video_metadata
)

print("=" * 65)
print("METADATA LABEL CHECK")
print("=" * 65)

print("FAKE :", metadata_fake)
print("REAL :", metadata_real)
print("TOTAL:", len(video_metadata))

METADATA LABEL CHECK
FAKE : 53
REAL : 53
TOTAL: 106


In [19]:
# ============================================================
# DEEPSHIELD-AI — VIDEO CONFIGURATION
# ============================================================

NUM_FRAMES = 16
FRAME_SIZE = 224

VIDEO_EXTENSIONS = (
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm"
)

print("=" * 65)
print("VIDEO CONFIGURATION")
print("=" * 65)

print("Frames per video :", NUM_FRAMES)
print("Frame size       :", FRAME_SIZE)
print("Supported formats:", VIDEO_EXTENSIONS)
print("Device            :", DEVICE)

VIDEO CONFIGURATION
Frames per video : 16
Frame size       : 224
Supported formats: ('.mp4', '.avi', '.mov', '.mkv', '.webm')
Device            : cuda


In [20]:
# ============================================================
# DEEPSHIELD-AI — DATA SPLITTING UTILITIES
# ============================================================

from sklearn.model_selection import train_test_split

print("Train/test splitting utilities imported.")

Train/test splitting utilities imported.


In [21]:
# ============================================================
# DEEPSHIELD-AI — STRATIFIED DATASET SPLIT
# ============================================================

metadata_indices = np.arange(len(video_metadata))
metadata_labels = np.array([
    item["label"]
    for item in video_metadata
])

# First: 70% train, 30% temporary
train_indices, temp_indices = train_test_split(
    metadata_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=metadata_labels
)

# Second: split temporary into 15% validation + 15% test
temp_labels = metadata_labels[temp_indices]

val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("=" * 65)
print("DATASET SPLIT")
print("=" * 65)

print("Train      :", len(train_indices))
print("Validation :", len(val_indices))
print("Test       :", len(test_indices))
print("Total      :", len(train_indices) + len(val_indices) + len(test_indices))

DATASET SPLIT
Train      : 74
Validation : 16
Test       : 16
Total      : 106


In [22]:
# ============================================================
# DEEPSHIELD-AI — VERIFY SPLIT BALANCE
# ============================================================

def print_split_distribution(name, indices):

    labels = [
        video_metadata[i]["label"]
        for i in indices
    ]

    fake_count = labels.count(0)
    real_count = labels.count(1)

    print(f"\n{name}")
    print("-" * 40)
    print("FAKE :", fake_count)
    print("REAL :", real_count)
    print("Total:", len(labels))


print("=" * 65)
print("CLASS DISTRIBUTION")
print("=" * 65)

print_split_distribution("TRAIN", train_indices)
print_split_distribution("VALIDATION", val_indices)
print_split_distribution("TEST", test_indices)

CLASS DISTRIBUTION

TRAIN
----------------------------------------
FAKE : 37
REAL : 37
Total: 74

VALIDATION
----------------------------------------
FAKE : 8
REAL : 8
Total: 16

TEST
----------------------------------------
FAKE : 8
REAL : 8
Total: 16


In [23]:
# ============================================================
# DEEPSHIELD-AI — VIDEO FRAME EXTRACTION
# ============================================================

def extract_video_frames(
    video_path,
    num_frames=NUM_FRAMES
):
    """
    Extract evenly spaced RGB frames from a video.

    Returns:
        frames: list of RGB numpy arrays
    """

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if total_frames <= 0:
        cap.release()
        raise RuntimeError(
            f"Invalid video: {video_path}"
        )

    frame_indices = np.linspace(
        0,
        total_frames - 1,
        num_frames,
        dtype=int
    )

    frames = []

    for frame_index in frame_indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_index)
        )

        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        raise RuntimeError(
            f"No frames extracted: {video_path}"
        )

    # If some frames failed, repeat the last valid frame
    while len(frames) < num_frames:
        frames.append(frames[-1].copy())

    frames = frames[:num_frames]

    return frames


print("Frame extraction function created successfully.")

Frame extraction function created successfully.


In [24]:
# ============================================================
# DEEPSHIELD-AI — FRAME EXTRACTION TEST
# ============================================================

test_index = int(train_indices[0])

test_video_path = video_metadata[
    test_index
]["video_path"]

test_label = video_metadata[
    test_index
]["label"]

test_frames = extract_video_frames(
    test_video_path,
    NUM_FRAMES
)

print("=" * 65)
print("FRAME EXTRACTION TEST")
print("=" * 65)

print("Video path    :", test_video_path)
print("Label         :", test_label)
print("Label name    :", "FAKE" if test_label == 0 else "REAL")
print("Frames        :", len(test_frames))
print("Frame shape   :", test_frames[0].shape)
print("Frame dtype   :", test_frames[0].dtype)

FRAME EXTRACTION TEST
Video path    : C:\Users\saksh\.cache\huggingface\hub\datasets--Hemgg--SDFVD-video-dataset\snapshots\11239a51248ad96a767460b7613cbf3b99b2f547\Real\v38.mp4
Label         : 1
Label name    : REAL
Frames        : 16
Frame shape   : (720, 1280, 3)
Frame dtype   : uint8


In [25]:
# ============================================================
# DEEPSHIELD-AI — IMAGE TRANSFORMS
# ============================================================

from PIL import Image
from torchvision import transforms

print("PIL and torchvision transforms imported.")

PIL and torchvision transforms imported.


In [26]:
# ============================================================
# DEEPSHIELD-AI — VIDEO FRAME PREPROCESSING
# ============================================================

video_transform = transforms.Compose([
    transforms.Resize((FRAME_SIZE, FRAME_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Video preprocessing transform created.")
print("Output size:", FRAME_SIZE, "x", FRAME_SIZE)

Video preprocessing transform created.
Output size: 224 x 224


In [27]:
# ============================================================
# DEEPSHIELD-AI — FINAL VIDEO DATASET CLASS
# ============================================================

from torch.utils.data import Dataset

class VideoDataset(Dataset):

    def __init__(
        self,
        metadata,
        indices,
        num_frames=NUM_FRAMES,
        transform=None
    ):
        self.metadata = metadata
        self.indices = list(indices)
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        metadata_index = self.indices[idx]

        item = self.metadata[metadata_index]

        video_path = item["video_path"]
        label = int(item["label"])

        # Extract frames
        frames = extract_video_frames(
            video_path,
            self.num_frames
        )

        processed_frames = []

        for frame in frames:

            # NumPy RGB → PIL
            image = Image.fromarray(frame)

            if self.transform is not None:
                image = self.transform(image)

            processed_frames.append(image)

        # [T, C, H, W]
        video_tensor = torch.stack(
            processed_frames
        )

        return video_tensor, label


print("Final VideoDataset created successfully.")

Final VideoDataset created successfully.


In [28]:
# ============================================================
# DEEPSHIELD-AI — CREATE VIDEO DATASETS
# ============================================================

train_dataset = VideoDataset(
    metadata=video_metadata,
    indices=train_indices,
    num_frames=NUM_FRAMES,
    transform=video_transform
)

val_dataset = VideoDataset(
    metadata=video_metadata,
    indices=val_indices,
    num_frames=NUM_FRAMES,
    transform=video_transform
)

test_dataset = VideoDataset(
    metadata=video_metadata,
    indices=test_indices,
    num_frames=NUM_FRAMES,
    transform=video_transform
)

print("=" * 65)
print("VIDEO DATASETS CREATED")
print("=" * 65)

print("Train      :", len(train_dataset))
print("Validation :", len(val_dataset))
print("Test       :", len(test_dataset))

VIDEO DATASETS CREATED
Train      : 74
Validation : 16
Test       : 16


In [29]:
# ============================================================
# DEEPSHIELD-AI — VIDEO DATASET SAMPLE CHECK
# ============================================================

sample_video, sample_label = train_dataset[0]

print("=" * 65)
print("VIDEO DATASET SAMPLE CHECK")
print("=" * 65)

print("Video tensor shape :", sample_video.shape)
print("Label              :", sample_label)
print(
    "Label name         :",
    "FAKE" if sample_label == 0 else "REAL"
)
print("Data type          :", sample_video.dtype)
print("Device             :", sample_video.device)

print("\nExpected:")
print("Shape  :", f"[{NUM_FRAMES}, 3, {FRAME_SIZE}, {FRAME_SIZE}]")
print("Type   : torch.float32")

VIDEO DATASET SAMPLE CHECK
Video tensor shape : torch.Size([16, 3, 224, 224])
Label              : 1
Label name         : REAL
Data type          : torch.float32
Device             : cpu

Expected:
Shape  : [16, 3, 224, 224]
Type   : torch.float32


In [30]:
# ============================================================
# DEEPSHIELD-AI — VIDEO DATALOADER
# ============================================================

from torch.utils.data import DataLoader

print("DataLoader imported successfully.")

DataLoader imported successfully.


In [31]:
# ============================================================
# DEEPSHIELD-AI — DATALOADER CONFIGURATION
# ============================================================

BATCH_SIZE = 2
NUM_WORKERS = 0

print("=" * 65)
print("DATALOADER CONFIGURATION")
print("=" * 65)

print("Batch size :", BATCH_SIZE)
print("Workers    :", NUM_WORKERS)


DATALOADER CONFIGURATION
Batch size : 2
Workers    : 0


In [32]:
# ============================================================
# DEEPSHIELD-AI — CREATE DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("=" * 65)
print("VIDEO DATALOADERS CREATED")
print("=" * 65)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

VIDEO DATALOADERS CREATED
Train batches: 37
Val batches  : 8
Test batches : 8


In [33]:
# ============================================================
# DEEPSHIELD-AI — BATCH CHECK
# ============================================================

videos, labels = next(iter(train_loader))

print("=" * 65)
print("VIDEO BATCH CHECK")
print("=" * 65)

print("Video batch shape :", videos.shape)
print("Labels shape      :", labels.shape)
print("Labels            :", labels)

print("\nVideo dtype       :", videos.dtype)
print("Labels dtype      :", labels.dtype)

VIDEO BATCH CHECK
Video batch shape : torch.Size([2, 16, 3, 224, 224])
Labels shape      : torch.Size([2])
Labels            : tensor([0, 1])

Video dtype       : torch.float32
Labels dtype      : torch.int64


In [34]:
# ============================================================
# DEEPSHIELD-AI — GPU BATCH TRANSFER CHECK
# ============================================================

videos_gpu = videos.to(DEVICE)
labels_gpu = labels.to(DEVICE)

print("=" * 65)
print("GPU BATCH TRANSFER CHECK")
print("=" * 65)

print("Video device :", videos_gpu.device)
print("Label device :", labels_gpu.device)

print("Video shape  :", videos_gpu.shape)
print("Label shape  :", labels_gpu.shape)

if DEVICE.type == "cuda":

    print("\nGPU memory allocated :",
          round(
              torch.cuda.memory_allocated() / 1024**2,
              2
          ),
          "MB"
    )

    print("GPU memory reserved  :",
          round(
              torch.cuda.memory_reserved() / 1024**2,
              2
          ),
          "MB"
    )

print("\nGPU batch transfer successful.")

GPU BATCH TRANSFER CHECK
Video device : cuda:0
Label device : cuda:0
Video shape  : torch.Size([2, 16, 3, 224, 224])
Label shape  : torch.Size([2])

GPU memory allocated : 18.38 MB
GPU memory reserved  : 22.0 MB

GPU batch transfer successful.


In [35]:
# ============================================================
# DEEPSHIELD-AI — GPU BATCH TRANSFER CHECK
# ============================================================

videos_gpu = videos.to(DEVICE)
labels_gpu = labels.to(DEVICE)

print("=" * 65)
print("GPU BATCH TRANSFER CHECK")
print("=" * 65)

print("Video device :", videos_gpu.device)
print("Label device :", labels_gpu.device)

print("Video shape  :", videos_gpu.shape)
print("Label shape  :", labels_gpu.shape)

if DEVICE.type == "cuda":

    print("\nGPU memory allocated :",
          round(
              torch.cuda.memory_allocated() / 1024**2,
              2
          ),
          "MB"
    )

    print("GPU memory reserved  :",
          round(
              torch.cuda.memory_reserved() / 1024**2,
              2
          ),
          "MB"
    )

print("\nGPU batch transfer successful.")

GPU BATCH TRANSFER CHECK
Video device : cuda:0
Label device : cuda:0
Video shape  : torch.Size([2, 16, 3, 224, 224])
Label shape  : torch.Size([2])

GPU memory allocated : 18.38 MB
GPU memory reserved  : 42.0 MB

GPU batch transfer successful.


In [36]:
# ============================================================
# DEEPSHIELD-AI — MODEL IMPORTS
# ============================================================

import torch.nn as nn
import torchvision.models as models

print("PyTorch model modules imported successfully.")

PyTorch model modules imported successfully.


In [37]:
# ============================================================
# DEEPSHIELD-AI — VIDEO MODEL CONFIGURATION
# ============================================================

NUM_CLASSES = 2

CNN_FEATURES = 512
LSTM_HIDDEN_SIZE = 256
LSTM_LAYERS = 1

DROPOUT = 0.3

print("=" * 65)
print("VIDEO MODEL CONFIGURATION")
print("=" * 65)

print("CNN              : ResNet18")
print("CNN features     :", CNN_FEATURES)
print("LSTM hidden size :", LSTM_HIDDEN_SIZE)
print("LSTM layers      :", LSTM_LAYERS)
print("Dropout          :", DROPOUT)
print("Classes          :", NUM_CLASSES)

VIDEO MODEL CONFIGURATION
CNN              : ResNet18
CNN features     : 512
LSTM hidden size : 256
LSTM layers      : 1
Dropout          : 0.3
Classes          : 2


In [38]:
# ============================================================
# DEEPSHIELD-AI — RESNET18 BACKBONE
# ============================================================

resnet = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Remove the original ImageNet classification layer
resnet.fc = nn.Identity()

print("=" * 65)
print("RESNET18 BACKBONE")
print("=" * 65)

print("Model loaded successfully.")
print("Output feature size:", CNN_FEATURES)

RESNET18 BACKBONE
Model loaded successfully.
Output feature size: 512


In [39]:
# ============================================================
# DEEPSHIELD-AI — RESNET18 BACKBONE
# ============================================================

resnet = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Remove the original ImageNet classification layer
resnet.fc = nn.Identity()

print("=" * 65)
print("RESNET18 BACKBONE")
print("=" * 65)

print("Model loaded successfully.")
print("Output feature size:", CNN_FEATURES)

RESNET18 BACKBONE
Model loaded successfully.
Output feature size: 512


In [41]:
# ============================================================
# DEEPSHIELD-AI — CREATE VIDEO MODEL
# SELF-CONTAINED VERSION
# ============================================================

import torch
import torch.nn as nn
import torchvision.models as models


class VideoResNet18LSTM(nn.Module):

    def __init__(
        self,
        cnn,
        cnn_features=512,
        hidden_size=256,
        num_layers=1,
        num_classes=2,
        dropout=0.3
    ):
        super().__init__()

        self.cnn = cnn

        self.lstm = nn.LSTM(
            input_size=cnn_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):

        # x shape:
        # [Batch, Frames, Channels, Height, Width]

        batch_size, num_frames, channels, height, width = x.shape

        # ----------------------------------------------------
        # CNN processes every frame
        # ----------------------------------------------------

        x = x.reshape(
            batch_size * num_frames,
            channels,
            height,
            width
        )

        features = self.cnn(x)

        # ----------------------------------------------------
        # Restore temporal dimension
        # ----------------------------------------------------

        features = features.reshape(
            batch_size,
            num_frames,
            -1
        )

        # ----------------------------------------------------
        # LSTM processes frame sequence
        # ----------------------------------------------------

        lstm_output, _ = self.lstm(features)

        # Last frame's temporal representation
        final_feature = lstm_output[:, -1, :]

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        final_feature = self.dropout(
            final_feature
        )

        output = self.classifier(
            final_feature
        )

        return output


# ============================================================
# CREATE RESNET18 BACKBONE
# ============================================================

resnet = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Remove ImageNet classifier
resnet.fc = nn.Identity()


# ============================================================
# CREATE VIDEO MODEL
# ============================================================

video_model = VideoResNet18LSTM(
    cnn=resnet,
    cnn_features=512,
    hidden_size=256,
    num_layers=1,
    num_classes=2,
    dropout=0.3
)


# Move model to GPU
video_model = video_model.to(DEVICE)


print("=" * 65)
print("VIDEO MODEL CREATED SUCCESSFULLY")
print("=" * 65)

print("Architecture : ResNet18 + LSTM")
print("Device       :", next(video_model.parameters()).device)

VIDEO MODEL CREATED SUCCESSFULLY
Architecture : ResNet18 + LSTM
Device       : cuda:0


In [42]:
# ============================================================
# DEEPSHIELD-AI — CREATE VIDEO MODEL
# SELF-CONTAINED VERSION
# ============================================================

import torch
import torch.nn as nn
import torchvision.models as models


class VideoResNet18LSTM(nn.Module):

    def __init__(
        self,
        cnn,
        cnn_features=512,
        hidden_size=256,
        num_layers=1,
        num_classes=2,
        dropout=0.3
    ):
        super().__init__()

        self.cnn = cnn

        self.lstm = nn.LSTM(
            input_size=cnn_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):

        # x shape:
        # [Batch, Frames, Channels, Height, Width]

        batch_size, num_frames, channels, height, width = x.shape

        # ----------------------------------------------------
        # CNN processes every frame
        # ----------------------------------------------------

        x = x.reshape(
            batch_size * num_frames,
            channels,
            height,
            width
        )

        features = self.cnn(x)

        # ----------------------------------------------------
        # Restore temporal dimension
        # ----------------------------------------------------

        features = features.reshape(
            batch_size,
            num_frames,
            -1
        )

        # ----------------------------------------------------
        # LSTM processes frame sequence
        # ----------------------------------------------------

        lstm_output, _ = self.lstm(features)

        # Last frame's temporal representation
        final_feature = lstm_output[:, -1, :]

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        final_feature = self.dropout(
            final_feature
        )

        output = self.classifier(
            final_feature
        )

        return output


# ============================================================
# CREATE RESNET18 BACKBONE
# ============================================================

resnet = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Remove ImageNet classifier
resnet.fc = nn.Identity()


# ============================================================
# CREATE VIDEO MODEL
# ============================================================

video_model = VideoResNet18LSTM(
    cnn=resnet,
    cnn_features=512,
    hidden_size=256,
    num_layers=1,
    num_classes=2,
    dropout=0.3
)


# Move model to GPU
video_model = video_model.to(DEVICE)


print("=" * 65)
print("VIDEO MODEL CREATED SUCCESSFULLY")
print("=" * 65)

print("Architecture : ResNet18 + LSTM")
print("Device       :", next(video_model.parameters()).device)

VIDEO MODEL CREATED SUCCESSFULLY
Architecture : ResNet18 + LSTM
Device       : cuda:0


In [43]:
# ============================================================
# DEEPSHIELD-AI — MODEL DEVICE CHECK
# ============================================================

model_device = next(
    video_model.parameters()
).device

print("=" * 65)
print("MODEL DEVICE CHECK")
print("=" * 65)

print("Model device :", model_device)
print("Target device:", DEVICE)

MODEL DEVICE CHECK
Model device : cuda:0
Target device: cuda


In [44]:
# ============================================================
# DEEPSHIELD-AI — GET TRAINING BATCH
# ============================================================

videos, labels = next(
    iter(train_loader)
)

videos = videos.to(DEVICE)
labels = labels.to(DEVICE)

print("=" * 65)
print("TRAINING BATCH")
print("=" * 65)

print("Videos shape :", videos.shape)
print("Labels shape :", labels.shape)
print("Labels       :", labels)
print("Device       :", videos.device)

TRAINING BATCH
Videos shape : torch.Size([2, 16, 3, 224, 224])
Labels shape : torch.Size([2])
Labels       : tensor([1, 0], device='cuda:0')
Device       : cuda:0


In [45]:
# ============================================================
# DEEPSHIELD-AI — FORWARD PASS
# ============================================================

video_model.eval()

with torch.no_grad():

    outputs = video_model(
        videos
    )

print("=" * 65)
print("FORWARD PASS")
print("=" * 65)

print("Input shape  :", videos.shape)
print("Output shape :", outputs.shape)
print("Raw output   :", outputs)

FORWARD PASS
Input shape  : torch.Size([2, 16, 3, 224, 224])
Output shape : torch.Size([2, 2])
Raw output   : tensor([[-0.2081, -0.0867],
        [-0.1710,  0.2759]], device='cuda:0')


In [46]:
# ============================================================
# DEEPSHIELD-AI — PREDICTION CHECK
# ============================================================

probabilities = torch.softmax(
    outputs,
    dim=1
)

predictions = torch.argmax(
    probabilities,
    dim=1
)

print("=" * 65)
print("PREDICTION CHECK")
print("=" * 65)

print("Probabilities:")
print(probabilities)

print("\nPredictions :", predictions)
print("True labels :", labels)

print("\nProbability sums:")
print(probabilities.sum(dim=1))

PREDICTION CHECK
Probabilities:
tensor([[0.4697, 0.5303],
        [0.3901, 0.6099]], device='cuda:0')

Predictions : tensor([1, 1], device='cuda:0')
True labels : tensor([1, 0], device='cuda:0')

Probability sums:
tensor([1., 1.], device='cuda:0')


In [47]:
# ============================================================
# DEEPSHIELD-AI — LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss()

print("=" * 65)
print("LOSS FUNCTION")
print("=" * 65)

print("Loss:", criterion)

LOSS FUNCTION
Loss: CrossEntropyLoss()


In [48]:
# ============================================================
# DEEPSHIELD-AI — LOSS TEST
# ============================================================

loss = criterion(
    outputs,
    labels
)

print("=" * 65)
print("LOSS CHECK")
print("=" * 65)

print("Output shape :", outputs.shape)
print("Label shape  :", labels.shape)
print("Loss         :", loss.item())

LOSS CHECK
Output shape : torch.Size([2, 2])
Label shape  : torch.Size([2])
Loss         : 0.787839949131012


In [49]:
# ============================================================
# DEEPSHIELD-AI — OPTIMIZER
# ============================================================

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(
    video_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("=" * 65)
print("OPTIMIZER CREATED")
print("=" * 65)

print("Optimizer    : AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay :", WEIGHT_DECAY)

OPTIMIZER CREATED
Optimizer    : AdamW
Learning rate: 0.0001
Weight decay : 0.0001


In [50]:
# ============================================================
# DEEPSHIELD-AI — VIDEO MODEL TRAINING
# ============================================================

import time
import copy
import os

NUM_EPOCHS = 10

best_val_accuracy = 0.0
best_model_state = None

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

MODEL_SAVE_DIR = os.path.join(
    "models",
    "video"
)

os.makedirs(
    MODEL_SAVE_DIR,
    exist_ok=True
)

BEST_MODEL_PATH = os.path.join(
    MODEL_SAVE_DIR,
    "video_resnet18_lstm_best.pth"
)

print("=" * 70)
print("DEEPSHIELD-AI — VIDEO MODEL TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    start_time = time.time()

    # ========================================================
    # TRAINING
    # ========================================================

    video_model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for videos, labels in train_loader:

        videos = videos.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = video_model(videos)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_train_loss += (
            loss.item() * labels.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            (predictions == labels)
            .sum()
            .item()
        )

        train_total += labels.size(0)

    train_loss = (
        running_train_loss / train_total
    )

    train_accuracy = (
        train_correct / train_total
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    video_model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for videos, labels in val_loader:

            videos = videos.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = video_model(videos)

            loss = criterion(
                outputs,
                labels
            )

            running_val_loss += (
                loss.item() * labels.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_correct += (
                (predictions == labels)
                .sum()
                .item()
            )

            val_total += labels.size(0)

    val_loss = (
        running_val_loss / val_total
    )

    val_accuracy = (
        val_correct / val_total
    )

    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history["train_loss"].append(
        train_loss
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_accuracy"].append(
        val_accuracy
    )

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    is_best = (
        val_accuracy > best_val_accuracy
    )

    if is_best:

        best_val_accuracy = val_accuracy

        best_model_state = copy.deepcopy(
            video_model.state_dict()
        )

        torch.save(
            {
                "model_state_dict": video_model.state_dict(),
                "best_val_accuracy": best_val_accuracy,
                "epoch": epoch + 1,
                "num_frames": NUM_FRAMES,
                "frame_size": FRAME_SIZE,
                "num_classes": NUM_CLASSES,
                "cnn_features": CNN_FEATURES,
                "lstm_hidden_size": LSTM_HIDDEN_SIZE
            },
            BEST_MODEL_PATH
        )

    elapsed = (
        time.time() - start_time
    ) / 60

    # ========================================================
    # EPOCH RESULT
    # ========================================================

    print()
    print("=" * 70)
    print(f"EPOCH {epoch + 1}/{NUM_EPOCHS}")
    print("=" * 70)

    print(
        f"Training Loss     : {train_loss:.4f}"
    )

    print(
        f"Training Accuracy : {train_accuracy:.4f}"
    )

    print(
        f"Validation Loss   : {val_loss:.4f}"
    )

    print(
        f"Validation Acc.   : {val_accuracy:.4f}"
    )

    print(
        f"Time              : {elapsed:.2f} minutes"
    )

    if is_best:
        print("★ BEST MODEL SAVED")

print()
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best Validation Accuracy : "
    f"{best_val_accuracy:.4f}"
)

print(
    "Best Model Saved At       :",
    BEST_MODEL_PATH
)

DEEPSHIELD-AI — VIDEO MODEL TRAINING

EPOCH 1/10
Training Loss     : 0.7600
Training Accuracy : 0.4865
Validation Loss   : 0.8532
Validation Acc.   : 0.1875
Time              : 0.49 minutes
★ BEST MODEL SAVED

EPOCH 2/10
Training Loss     : 0.7050
Training Accuracy : 0.5000
Validation Loss   : 0.9672
Validation Acc.   : 0.1250
Time              : 0.47 minutes

EPOCH 3/10
Training Loss     : 0.6145
Training Accuracy : 0.7162
Validation Loss   : 1.1225
Validation Acc.   : 0.1250
Time              : 0.47 minutes

EPOCH 4/10
Training Loss     : 0.4792
Training Accuracy : 0.7838
Validation Loss   : 1.3211
Validation Acc.   : 0.3125
Time              : 0.47 minutes
★ BEST MODEL SAVED

EPOCH 5/10
Training Loss     : 0.4506
Training Accuracy : 0.7838
Validation Loss   : 1.1689
Validation Acc.   : 0.3750
Time              : 0.46 minutes
★ BEST MODEL SAVED

EPOCH 6/10
Training Loss     : 0.4447
Training Accuracy : 0.8649
Validation Loss   : 1.6836
Validation Acc.   : 0.1875
Time              : 0

In [51]:
# ============================================================
# DEEPSHIELD-AI — LOAD BEST VIDEO MODEL
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

video_model.load_state_dict(
    checkpoint["model_state_dict"]
)

video_model = video_model.to(DEVICE)
video_model.eval()

print("=" * 70)
print("BEST VIDEO MODEL LOADED")
print("=" * 70)

print("Checkpoint epoch        :", checkpoint["epoch"])
print("Best validation accuracy:", checkpoint["best_val_accuracy"])
print("Model path              :", BEST_MODEL_PATH)

BEST VIDEO MODEL LOADED
Checkpoint epoch        : 5
Best validation accuracy: 0.375
Model path              : models\video\video_resnet18_lstm_best.pth


In [52]:
# ============================================================
# DEEPSHIELD-AI — BEST CHECKPOINT FORWARD TEST
# ============================================================

videos, labels = next(iter(val_loader))

videos = videos.to(DEVICE)
labels = labels.to(DEVICE)

with torch.no_grad():
    outputs = video_model(videos)

probabilities = torch.softmax(
    outputs,
    dim=1
)

predictions = torch.argmax(
    probabilities,
    dim=1
)

print("=" * 70)
print("BEST CHECKPOINT VERIFICATION")
print("=" * 70)

print("Input shape :", videos.shape)
print("Output shape:", outputs.shape)

print("\nProbabilities:")
print(probabilities)

print("\nPredictions :", predictions.cpu().numpy())
print("True labels :", labels.cpu().numpy())

BEST CHECKPOINT VERIFICATION
Input shape : torch.Size([2, 16, 3, 224, 224])
Output shape: torch.Size([2, 2])

Probabilities:
tensor([[0.6871, 0.3129],
        [0.7971, 0.2029]], device='cuda:0')

Predictions : [0 0]
True labels : [0 1]
